# Contrastive Probe Inference

Runs the spatial-grounding probe as a factorial of control conditions over two
scene sources, logging both the executable action and a continuous readout.

**Continuous readout.** OpenVLA emits one token per action dimension and decodes
it to a bin centre, so the executable action is quantised. The lateral bin is
roughly 1e-3 wide while observed lateral predictions sit between 1e-4 and 4e-3
of zero, which put the earlier median paired difference at exactly zero in every
stratum with several pairs bit-identical between the two instructions. Alongside
the argmax action, each prediction now records the expected bin centre under the
model's own distribution over action tokens, which resolves differences smaller
than one bin. 

**Control conditions.** A model mapping the token `left` to a leftward action without consulting the
image reproduces the expected sign flip exactly, and a model whose lateral output
ignores the image produces no difference for reasons unrelated to language. Each
condition varies one factor: mirroring reverses the lateral axis with the
instruction held fixed, pairing an instruction with another scene removes its
referent while leaving the language intact, and removing the spatial term gives a
within-scene reference.

**Two scene sources.** The experiments run on the `constructed` scenes, which hold
two instances of the target noun and whose geometry is recorded rather than
inferred. The `bridge` scenes are the unaltered frames harvested into the validation
role, and they answer whether a
constructed-set finding transfers to data that was never edited. The two sets are
disjoint by construction, so no frame appears in both.

The constructed source is read from the frozen `evaluation_set.csv` written by
Notebook 03, not from the constructed manifest. The manifest holds everything ever
built, including scenes that failed the approval screen and the overbuild beyond the
target, and probing those would spend GPU time on stimuli the analysis excludes
while making the set that was measured depend on when the probe happened to run.

Output is `probe_predictions.csv` under the v2 generation, written fresh. Earlier
logs are not migrated: they were collected over a different scene set under a
different design, and copying them forward would mix two populations in one file.

## 0. Dependencies

OpenVLA-7B loads only against the pinned dependency set, which is installed here
rather than inherited from a session that happened to run Notebook 01 first. A
runtime that has not installed it keeps the Colab defaults, whose transformers 5.x
releases no longer expose `AutoModelForVision2Seq`, and the mismatch surfaces as an
import error inside [model.py](model.py) in the section below.

The pins are restated in this notebook, mirroring `requirements-colab.txt` and
Notebook 01, because they have to be installed before the repository is cloned and
before any pinned package is imported. A module already imported keeps its own code
for the life of the interpreter, so installing first is what makes the replacement
effective.

Three properties of the install matter:

- The interpreter version is asserted before anything else, because it decides
  whether the pinned set can be installed at all. `tokenizers==0.19.1`, which
  OpenVLA's remote modelling code requires, publishes no wheel beyond CPython 3.12.
  Pin the Colab runtime version to 2026.07 under Runtime > Change runtime type.
- `--only-binary=:all:` forbids a source build, so a missing wheel stops the install
  instead of failing late inside a compiler and leaving the pre-installed versions
  in place.
- torch is never reinstalled. The Colab build is matched to its CUDA driver, and
  overriding it tends to break GPU support.

The verification that closes the cell reads both the versions recorded on disk and
the version of any pinned module already imported. Those disagree when a package was
imported before the install, which is the one condition a session restart resolves,
and it is named here rather than left to surface as an obscure failure inside the
model load.

`pip` may report dependency conflicts against pre-installed Colab packages such as
`datasets` and `diffusers`, which want a newer `huggingface_hub`. Those packages are
not used anywhere in this pipeline and the conflict is expected.

In [3]:
import subprocess
import sys
from importlib.metadata import version

REQUIRED_PYTHON = (3, 12)          # highest version with wheels for the pinned set
COLAB_RUNTIME_VERSION = '2026.07'  # last runtime version shipping that interpreter

# Mirrors requirements-colab.txt and the install cell of Notebook 01, restated here
# because this cell runs before the repository is cloned. The trio at the top is the
# OpenVLA authors' known-good set; newer releases cause a misleading "requires
# prismatic" load error, and the 5.x line withdraws the interface model.py loads
# through. protobuf is bounded above as well as below, mirroring data.PROTOBUF_SPEC,
# and installed without --upgrade so a conforming runtime is left in place.
PINS = [
    'transformers==4.40.1',
    'tokenizers==0.19.1',
    'timm==0.9.10',
    'huggingface_hub==0.23.4',
    'accelerate==0.30.1',
    'bitsandbytes>=0.45.0',
    'protobuf>=6.31.1,<7',
]
EXPECTED = {
    'transformers': '4.40.1',
    'tokenizers': '0.19.1',
    'timm': '0.9.10',
    'huggingface_hub': '0.23.4',
    'accelerate': '0.30.1',
}

assert sys.version_info[:2] == REQUIRED_PYTHON, (
    f'Python {sys.version_info.major}.{sys.version_info.minor} is active, but the pinned '
    f'dependencies require Python {REQUIRED_PYTHON[0]}.{REQUIRED_PYTHON[1]}. Set Runtime > '
    f'Change runtime type > Runtime version to {COLAB_RUNTIME_VERSION}, then reconnect and '
    f'run this notebook from the top.'
)

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--only-binary=:all:', *PINS],
    check=True,
)

installed = {name: version(name) for name in EXPECTED}
for name, found in installed.items():
    print(f'{name:16s} {found}')

mismatched = {n: v for n, v in installed.items() if v != EXPECTED[n]}
assert not mismatched, (
    'Installed versions differ from the pin: '
    + ', '.join(f'{n} {v} (expected {EXPECTED[n]})' for n, v in mismatched.items())
    + '. Re-run this cell and check the install reported no error.'
)

# A module imported before the install still holds the previous code, which no
# further install can replace in this interpreter.
imported = {n: getattr(sys.modules[n], '__version__', '') for n in EXPECTED
            if n in sys.modules}
stale = {n: v for n, v in imported.items() if v and v != EXPECTED[n]}
assert not stale, (
    'These packages were imported before the install and the session is still '
    'running the earlier code: '
    + ', '.join(f'{n} {v} (expected {EXPECTED[n]})' for n, v in stale.items())
    + '. Restart the session (Runtime > Restart session) and run from the top.'
)
print(f'\nPython {sys.version.split()[0]}; pinned dependency set active')

transformers     4.40.1
tokenizers       0.19.1
timm             0.9.10
huggingface_hub  0.23.4
accelerate       0.30.1


AssertionError: These packages were imported before the install and the session is still running the earlier code: transformers 5.13.1 (expected 4.40.1), tokenizers 0.22.2 (expected 0.19.1), huggingface_hub 1.23.0 (expected 0.23.4), accelerate 1.14.0 (expected 0.30.1). Restart the session (Runtime > Restart session) and run from the top.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['HF_HOME'] = '/content/drive/MyDrive/openvla_cache/hf'
CACHE_DIR = '/content/drive/MyDrive/openvla_cache/v2/bridge'
CONSTRUCTED_DIR = '/content/drive/MyDrive/openvla_cache/v2/constructed'
# One log per generation. The v2 generation harvested its frames fresh and built
# its constructed set fresh, so its predictions live in their own file rather than
# being appended to a log collected over a different scene set.
PROBE_CSV = '/content/drive/MyDrive/openvla_cache/v2/probe_predictions.csv'
print('cache       ->', CACHE_DIR)
print('constructed ->', CONSTRUCTED_DIR)
print('log         ->', PROBE_CSV)

## 2. Import the code

Clones the project code from GitHub into the runtime and imports the loader,
inference, control, and logging functions from there, so the code always matches
the pushed commit.

In [ ]:
import sys, os, importlib, subprocess

REPO_URL = 'https://github.com/LewisTL/ECS8056.git'
BRANCH = 'master'
REPO_DIR = '/content/ECS8056'

def sync_repo():
    """Clone or hard-refresh the repository so it matches origin/BRANCH."""
    token = os.environ.get('GITHUB_TOKEN', '')
    url = REPO_URL.replace('https://', f'https://{token}@') if token else REPO_URL
    if os.path.isdir(os.path.join(REPO_DIR, '.git')):
        subprocess.run(['git', '-C', REPO_DIR, 'remote', 'set-url', 'origin', url],
                       check=True)
        subprocess.run(['git', '-C', REPO_DIR, 'fetch', '--quiet', '--depth', '1',
                        'origin', BRANCH], check=True)
        subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', '--quiet',
                        f'origin/{BRANCH}'], check=True)
    else:
        subprocess.run(['git', 'clone', '--quiet', '--depth', '1', '--branch',
                        BRANCH, url, REPO_DIR], check=True)
    return subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', '--short', 'HEAD'],
                          capture_output=True, text=True).stdout.strip()


commit = sync_repo()
module_dir = REPO_DIR
if module_dir not in sys.path:
    sys.path.insert(0, module_dir)
for _m in ('action_bins', 'prediction_log', 'model', 'data', 'controls', 'compose_scenes', 'export_pairs', 'detect_duplicates', 'analysis',):
    sys.modules.pop(_m, None)
importlib.invalidate_caches()

from model import (load_openvla, predict_action, predict_action_dist,
                   describe_action_space, verify_readout, run_metadata,
                   append_prediction_log)
from prediction_log import ensure_readable
from data import (SPLIT_VALIDATION, load_manifest, make_pair, term_axis,
                  term_axis_index)
from controls import (plan_stimuli, build_scene_swap, apply_image_transform,
                      strip_spatial_term, DEFAULT_CONDITIONS)
from compose_scenes import (evaluation_scenes, load_constructed_manifest,
                            IMAGE_X_TO_LATERAL_SIGN)
import analysis
print(f'imported project modules from {module_dir} @ {commit}')

## 3. Load OpenVLA-7B

In [ ]:
processor, vla, compute_dtype = load_openvla(quantize_4bit=True, precision='bf16')
meta = run_metadata(compute_dtype)
print(meta)

## 4. Action space and readout verification

The continuous readout reimplements the decoding OpenVLA performs inside
`predict_action`, against constants that live in its remote modelling code and
can move between revisions. Those constants are printed here rather than
assumed, and the reimplementation is then required to reproduce the executable
action exactly on real inputs before any continuous value is used.

`bin_width` is the resolution floor of the argmax readout on each dimension: two
predictions closer together than this cannot differ in the executable action, no
matter how differently the model treats them. It is the number the earlier null
results should have been read against, and it sets the equivalence bound in the
analysis notebook.

In [ ]:
space = describe_action_space(vla)
for key, value in space.items():
    if isinstance(value, list):
        print(f'{key:20} ' + ' '.join(f'{v:+.5f}' if isinstance(v, float) else str(v)
                                      for v in value))
    else:
        print(f'{key:20} {value}')
print()
print(f"lateral (dx) bin width: {space['bin_width'][0]:.6f}")

### Correctness gate

Fifty real scenes are run through both paths. Any disagreement raises, because a
continuous value derived from misidentified constants would be worse than no
continuous value at all: it would look plausible and be wrong.

In [ ]:
from PIL import Image

# The validation role: unaltered frames, held disjoint from the base frames the
# constructed stimuli were composited from.
rows = [r for r in load_manifest(CACHE_DIR) if r['split'] == SPLIT_VALIDATION]
gate_samples = []
for row in rows:
    made = make_pair(row.get('instruction', ''))
    if made is None:
        continue
    gate_samples.append((Image.open(os.path.join(CACHE_DIR, row['image_path'])),
                         row['instruction']))
    if len(gate_samples) >= 50:
        break

gate = verify_readout(processor, vla, gate_samples)
assert gate['matched'] == gate['checked'], gate
print('continuous readout verified against predict_action on '
      f"{gate['checked']} scenes")

## 5. Assemble the probe set

Both scene sources are expanded into the same record shape, so the probe loop
and the log schema do not depend on where a scene came from.

The constructed scenes come from the frozen `evaluation_set.csv`, so exactly the
approved and selected set is measured. The bridge scenes are the frames harvested
into the validation role, which is disjoint from the base frames the constructed
stimuli were built on.

For `bridge` scenes the axis comes from the spatial term, and there is no
recorded geometry, so no directional expectation is attached. For `constructed`
scenes the axis is lateral by construction and `expected_sign` is taken from the
manifest geometry, converted from image coordinates by
`IMAGE_X_TO_LATERAL_SIGN`. That constant is the one unresolved link between image
position and action sign; it is applied here in a single place, and the mirror
control establishes it empirically in the analysis notebook.

In [ ]:
probe_set = []

# --- Bridge scenes: the unaltered validation source --------------------------
for row in rows:
    made = make_pair(row['instruction'])
    if made is None:
        continue
    term, swapped = made
    axis_index = term_axis_index(term)
    if axis_index is None:
        # Scene-dependent relations ("closer to the plate") have no axis that can
        # be fixed before seeing the scene, so they carry no expectation.
        continue
    # category_manual overrides the heuristic category when a scene has been
    # manually reviewed; downstream code follows the same fallback.
    category = row.get('category_manual') or row.get('category', 'other')
    probe_set.append({
        'scene_source': 'bridge',
        'scene_id': f"b{int(row['episode_index']):06d}",
        'pair_id': f"ep{int(row['episode_index']):06d}_{term}",
        'spatial_term': term,
        'axis': term_axis(term),
        'axis_index': axis_index,
        'configuration': '',
        'expected_sign': 0,
        'target_sign_a': 0,
        'target_sign_b': 0,
        'category': category,
        'feasible_both': row.get('feasible_both', 'unreviewed'),
        'duplicate_target': row.get('duplicate_target', 'unreviewed'),
        'image_path': os.path.join(CACHE_DIR, row['image_path']),
        'instr_a': row['instruction'],
        'instr_b': swapped,
    })

# --- Constructed scenes: the experimental source, as frozen ------------------
# Read from evaluation_set.csv rather than the manifest, so the scenes measured are
# the approved and selected ones and cannot change as construction or screening
# continues.
constructed = evaluation_scenes(CONSTRUCTED_DIR)
for scene in constructed:
    probe_set.append({
        'scene_source': 'constructed',
        'scene_id': scene['construct_id'],
        'pair_id': scene['construct_id'],
        'spatial_term': scene['spatial_term'],
        'axis': 'lateral',
        'axis_index': 0,
        'configuration': scene['configuration'],
        'expected_sign': int(scene['expected_sign_image']) * IMAGE_X_TO_LATERAL_SIGN,
        # The side each instruction's own target sits on. Scoring each
        # instruction against its own target, rather than scoring the pair
        # against their relative order, is what separates scene grounding from
        # a fixed word-to-direction mapping on the same-side configurations.
        'target_sign_a': int(scene['target_sign_a_image']) * IMAGE_X_TO_LATERAL_SIGN,
        'target_sign_b': int(scene['target_sign_b_image']) * IMAGE_X_TO_LATERAL_SIGN,
        'category': 'constructed',
        'feasible_both': 'yes',
        'duplicate_target': 'yes',
        'image_path': os.path.join(CONSTRUCTED_DIR, scene['image_path']),
        'instr_a': scene['instr_a'],
        'instr_b': scene['instr_b'],
    })

from collections import Counter
print(f'{len(probe_set)} scenes: ' + str(dict(Counter(
    p['scene_source'] for p in probe_set))))
print('bridge axes:', dict(Counter(p['axis'] for p in probe_set
                                   if p['scene_source'] == 'bridge')))
print('constructed configurations:', dict(Counter(
    p['configuration'] for p in probe_set if p['scene_source'] == 'constructed')))
if not constructed:
    built = (len(load_constructed_manifest(CONSTRUCTED_DIR))
             if os.path.isfile(os.path.join(CONSTRUCTED_DIR,
                                            'constructed_manifest.csv')) else 0)
    print(f'\nNo frozen evaluation set found ({built} scenes built). Screen and '
          'freeze in Notebook 03 before probing: the experiments run on the frozen '
          'set, and the bridge source alone supports only the instrument check and '
          'the validation analysis.')

## 6. Expand the condition factorial

Every scene is expanded into the predictions its applicable conditions require.
A condition is skipped, not approximated, when its precondition fails: the mirror
conditions need a lateral term because a horizontal flip leaves depth and
vertical relations unchanged, and the term-stripped conditions need a removal
that leaves a well-formed instruction.

Refusals are counted and reported here. A truncated prompt would change the
prediction for reasons unrelated to the spatial term, so producing one would
quietly corrupt the within-scene reference; skipping instead means the neutral
conditions cover a subset of scenes, which the analysis accounts for.

The swapped-scene assignment is a derangement, so no scene is ever paired with
its own image and the control cannot silently degrade into the baseline. It is
built separately per scene source, keeping the replacement image in the same
visual distribution as the original.

In [ ]:
swap_map = {}
for source in ('bridge', 'constructed'):
    ids = [p['scene_id'] for p in probe_set if p['scene_source'] == source]
    if len(ids) >= 2:
        swap_map.update(build_scene_swap(ids, seed=0))

image_lookup = {p['scene_id']: p['image_path'] for p in probe_set}

work = []
refused_strip = 0
for p in probe_set:
    stimuli = plan_stimuli(p, swap_map=swap_map, conditions=DEFAULT_CONDITIONS)
    if strip_spatial_term(p['instr_a'], p['spatial_term']) is None:
        refused_strip += 1
    for stim in stimuli:
        work.append((p, stim))

print(f'{len(work)} predictions across {len(probe_set)} scenes')
print('per condition:', dict(Counter(s.condition for _, s in work)))
print('per scene source:', dict(Counter(p['scene_source'] for p, _ in work)))
print(f'\nterm removal refused on {refused_strip}/{len(probe_set)} scenes '
      f'({refused_strip / max(len(probe_set), 1):.1%}); those scenes contribute '
      'no neutral reference')

for p, stim in work[:8]:
    print(f"  [{p['scene_id']}] {stim.condition:16} {stim.role} "
          f"{stim.image_transform:14} :: {stim.instruction}")

## 7. Start the log

The log is written fresh for this generation. Nothing is migrated from an earlier
one: those predictions were collected over a different scene set, and copying them
forward would put two populations in one file with only a column to tell them
apart, which the analysis would then have to remember to split on everywhere.

The file is created with the complete header from `probe_log_fields()` in
[prediction_log.py](prediction_log.py), the single declaration of the log's
columns, which the probe loop and the analysis notebooks also read. A log whose
header is narrower than the rows later appended to it is not a CSV any reader can
parse, and the failure appears far from its cause: every append succeeds, and the
file breaks only when the analysis first tries to read it.

In [ ]:
import pandas as pd
from prediction_log import PROBE_EXTRA_FIELDS, probe_log_fields

PROBE_LOG_FIELDS = probe_log_fields()
print(f'v4 schema: {len(PROBE_LOG_FIELDS)} columns')

os.makedirs(os.path.dirname(PROBE_CSV), exist_ok=True)
if os.path.exists(PROBE_CSV):
    existing = pd.read_csv(PROBE_CSV)
    print(f'{PROBE_CSV} holds {len(existing)} rows; the probe below resumes into it')
    print('by scene source:', dict(existing['scene_source'].value_counts()))
else:
    pd.DataFrame(columns=PROBE_LOG_FIELDS).to_csv(PROBE_CSV, index=False)
    print(f'created {PROBE_CSV} with the full header')

## 7b. Check the log is readable

A log written before the header was declared in full can hold rows wider than its
own header, which no CSV reader will parse: the error names a line number and
nothing else. `ensure_readable` reports the field counts actually present and
rebuilds the file when they differ, reading each row under the schema matching its
width so that no value moves to a different column.

It refuses to rewrite a log holding rows it cannot account for, since those would
be dropped, so the repair cannot lose data. Nothing is re-run on GPU: this is a
file-format repair, and a no-op on a log that is already consistent.

The analysis notebooks call the same function through `load_inputs`, so the log is
checked wherever it is read rather than only here.

In [ ]:
ensure_readable(PROBE_CSV)

## 8. Run the probe

One deterministic prediction per work item, each logged with its condition, role,
scene source, and the axis its comparison will be read on.

Restart-safe: the resume key is
`(scene_source, pair_id, frame, condition, role)`, which extends the v3 key with
the two new factors. Role `n` covers the single-prediction term-stripped
conditions, which have no opposite.

`sample_idx` is fixed at 0 under the deterministic decoding strategy; the column
exists so the schema does not change if repeated sampling is added later.

In [ ]:
import csv
from PIL import Image

FRAME = 'initial'

done = set()
if os.path.exists(PROBE_CSV):
    with open(PROBE_CSV, newline='') as f:
        for r in csv.DictReader(f):
            done.add((r.get('scene_source', 'bridge'), r['pair_id'],
                      r.get('frame') or 'initial',
                      r.get('condition', 'baseline'), r['role']))
    print(f'resuming: {len(done)} predictions already logged')

image_cache = {}
def load_image(scene_id):
    if scene_id not in image_cache:
        image_cache[scene_id] = Image.open(image_lookup[scene_id]).convert('RGB')
    return image_cache[scene_id]

run = 0
for i, (p, stim) in enumerate(work):
    key = (p['scene_source'], p['pair_id'], FRAME, stim.condition, stim.role)
    if key in done:
        continue
    image = apply_image_transform(stim.image_transform,
                                  load_image(stim.image_scene_id))
    readout = predict_action_dist(processor, vla, image, stim.instruction,
                                  compute_dtype)
    # Keyed by PROBE_EXTRA_FIELDS so the row cannot carry a column the declared
    # header lacks, and asserted rather than trusted.
    extra = {
        'scene_id': p['scene_id'],
        'pair_id': p['pair_id'],
        'role': stim.role,
        'frame': FRAME,
        'scene_source': p['scene_source'],
        'condition': stim.condition,
        'image_transform': stim.image_transform,
        'image_scene_id': stim.image_scene_id,
        'configuration': p['configuration'],
        'expected_sign': p['expected_sign'],
        'target_sign_a': p['target_sign_a'],
        'target_sign_b': p['target_sign_b'],
        'spatial_term': p['spatial_term'],
        'axis': p['axis'],
        'axis_index': p['axis_index'],
        'category': p['category'],
        'feasible_both': p['feasible_both'],
        'duplicate_target': p['duplicate_target'],
        'sample_idx': 0,
    }
    assert list(extra) == PROBE_EXTRA_FIELDS, 'log fields drifted from the schema'
    append_prediction_log(PROBE_CSV, readout.action, stim.instruction, meta,
                          readout=readout, **extra)
    done.add(key)
    run += 1
    if run % 100 == 0:
        print(f'{run} predictions run ({i + 1}/{len(work)} work items seen)')

print(f'probe complete: {run} new predictions -> {PROBE_CSV}')

## 9. Determinism check

Twenty stimuli already in the log are re-predicted. Decoding is greedy with fixed
seeds, so the repeats must agree exactly. Establishing that here means any
nonzero difference in the analysis is attributable to the manipulation rather
than to run-to-run variation, which is what lets differences of the size the
continuous readout resolves be taken seriously at all.

In [ ]:
import numpy as np

repeats = []
for p, stim in work[:20]:
    image = apply_image_transform(stim.image_transform,
                                  load_image(stim.image_scene_id))
    again = predict_action_dist(processor, vla, image, stim.instruction,
                                compute_dtype)
    repeats.append((p['scene_id'], stim.condition, stim.role,
                    again.action, again.expected))

log = pd.read_csv(PROBE_CSV)
worst_action, worst_cont, compared = 0.0, 0.0, 0
for scene_id, condition, role, action, expected in repeats:
    match = log[(log['scene_id'] == scene_id) & (log['condition'] == condition)
                & (log['role'] == role)]
    if match.empty:
        continue
    logged_a = match[[f'a{i}' for i in range(7)]].iloc[0].to_numpy(dtype=float)
    logged_c = match[[f'c{i}' for i in range(7)]].iloc[0].to_numpy(dtype=float)
    worst_action = max(worst_action, float(np.max(np.abs(logged_a - action))))
    if np.isfinite(logged_c).all():
        worst_cont = max(worst_cont, float(np.max(np.abs(logged_c - expected))))
    compared += 1

print(f'{compared} stimuli re-predicted')
print(f'largest argmax difference:     {worst_action:.3e}')
print(f'largest continuous difference: {worst_cont:.3e}')
assert worst_action == 0.0 and worst_cont == 0.0, (
    'repeated identical inputs disagreed; decoding is not deterministic and no '
    'difference measured downstream can be attributed to the manipulation')
print('deterministic')

## 10. Instrument check

The gate that decides whether the language measurement can mean anything.

With the instruction held fixed, mirroring the image reverses the lateral axis of
the scene. A model that reads lateral position at all must change the sign of its
lateral output. If it does not, the visual channel is not live on this axis, and
a null on the language comparisons carries no information about spatial language:
it would follow from the model ignoring the image entirely. The term-stripped
variant is the cleaner form, since it isolates object grounding from any
influence of the spatial word.

This check is not itself evidence of spatial language grounding. It establishes
the necessary condition that makes the rest of the analysis interpretable, and it
identifies the lateral axis empirically, which the earlier ground-truth pilot
failed to do.

In [ ]:
log = pd.read_csv(PROBE_CSV)
lateral = log[(log['scene_source'] == 'bridge') & (log['axis_index'] == 0)
              & log['c0'].notna()]
check = analysis.mirror_check(lateral)

for label in ('neutral', 'term'):
    result = check.get(label, {})
    if not result.get('n'):
        print(f'{label}: no paired mirror predictions yet')
        continue
    print(f"[{label}] n={result['n']}")
    print(f"  sign flips under mirroring : {result['flip_rate']:.1%}")
    print(f"  identical to original      : {result['identical_rate']:.1%}")
    print(f"  mean |lateral| original    : {result['mean_abs_original']:.5f}")
    print(f"  mean |change|              : {result['mean_abs_change']:.5f}")
    print(f"  antisymmetry (0 if exact reversal): "
          f"median={result['antisymmetry']['median']:+.5f} "
          f"p={result['antisymmetry']['p_value']:.3g}")
    print(f"  invariance   (0 if ignored)      : "
          f"median={result['invariance']['median']:+.5f} "
          f"p={result['invariance']['p_value']:.3g}")

print('\nRead: a high flip rate with an antisymmetry median near zero means the '
      'lateral channel tracks the scene, and the language comparisons are '
      'interpretable. A near-zero flip rate with an invariance median near zero '
      'means the output ignores the image, and no language conclusion can be '
      'drawn from this axis.')